# ResNet50 Image Classification Experiment

This separate experiment assigns each image the class of its first annotated box. It is a heuristic for image-level labels and can miss other damage classes in the same image. The notebook copies images into class folders and evaluates classification on the test split.

In [ ]:
import os
from pathlib import Path

from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("Set ROBOFLOW_API_KEY before starting Jupyter.")
project = Roboflow(api_key=api_key).workspace("ozu").project("xview2")
dataset = project.version(4).download("yolov8")
data_yaml = Path(dataset.location).resolve() / "data.yaml"
if not data_yaml.is_file():
    raise FileNotFoundError(data_yaml)


In [ ]:
# 2. المكتبات المطلوبة
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 3. الجهاز المستخدم (GPU أو CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 4. دالة لتنظيم الصور داخل مجلدات الفئات بناءً على ملفات التسمية
def organize_images_by_labels(base_path):
    images_dir = os.path.join(base_path, "images")
    labels_dir = os.path.join(base_path, "labels")

    organized_dir = os.path.join(base_path, "organized")
    if not os.path.exists(organized_dir):
        os.makedirs(organized_dir)

    for label_file in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, label_file)
        with open(label_path, 'r') as f:
            lines = f.readlines()
            if not lines:
                continue
            first_line = lines[0].strip()
            class_id = first_line.split()[0]

        # جرب امتداد الصورة png ثم jpg
        img_name_png = os.path.splitext(label_file)[0] + ".png"
        img_name_jpg = os.path.splitext(label_file)[0] + ".jpg"

        img_path = os.path.join(images_dir, img_name_png)
        if not os.path.exists(img_path):
            img_path = os.path.join(images_dir, img_name_jpg)
            if not os.path.exists(img_path):
                print(f"تحذير: الصورة غير موجودة لـ {label_file}")
                continue

        class_dir = os.path.join(organized_dir, class_id)
        if not os.path.exists(class_dir):
            os.makedirs(class_dir)

        shutil.copy(img_path, os.path.join(class_dir, os.path.basename(img_path)))

    print(f"تم تنظيم الصور في {base_path}")

# 5. نفذ التنظيم لمجلدات train, valid, test
for split in ["train", "valid", "test"]:
    organize_images_by_labels(os.path.join(dataset.location, split))

# 6. إعداد تحويل الصور
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# 7. تحميل البيانات من المجلدات المنظمة
train_dir = os.path.join(dataset.location, "train", "organized")
val_dir = os.path.join(dataset.location, "valid", "organized")
test_dir = os.path.join(dataset.location, "test", "organized")

train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)
test_data = datasets.ImageFolder(test_dir, transform=transform)
if train_data.class_to_idx != test_data.class_to_idx:
    raise ValueError("Training and test split class mappings must match.")

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

# 8. بناء نموذج ResNet50 مع تعديل الطبقة الأخيرة وعدد الفئات
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False  # تجميد الطبقات السابقة

num_classes = len(train_data.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# 9. إعداد الخسارة والمُحسن
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.002)

# 10. تدريب النموذج
epochs = 20
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"📘 Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Accuracy: {accuracy:.2f}%")

# 11. تقييم النموذج
def evaluate_model(model, dataloader, class_names):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n📊 Classification Report:\n")
    print(classification_report(all_labels, all_preds, labels=list(range(len(class_names))), target_names=class_names, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title("🌀 Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# 12. تشغيل التقييم على بيانات الاختبار
evaluate_model(model, test_loader, train_data.classes)
